# System Evaluation — Requirement #8
## Dataset: beir/quora/test

This notebook computes the four required IR evaluation metrics for every representation model:
- **MAP** — Mean Average Precision
- **Recall@10** — fraction of relevant documents found in top-10
- **Precision@10** — fraction of top-10 results that are relevant
- **nDCG@10** — Normalized Discounted Cumulative Gain

**Evaluation is run in two conditions:**
1. **Before** additional features (NER reranking OFF, Cluster reranking OFF)
2. **After** additional features (NER reranking ON, Cluster reranking ON)

**Models evaluated (5 models):**
- TF-IDF
- BM25
- BERT
- Hybrid Parallel (score fusion)
- Hybrid Serial (BM25 retrieve → BERT re-rank)

In [ ]:
import requests
import ir_datasets
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────
DATASET_NAME      = 'beir/quora/test'
SEARCH_API_URL    = 'http://127.0.0.1:8003/search/'
TOP_K             = 10
QUERY_SAMPLE_SIZE = 500    # None = all 10,000 queries (slower)
BEST_K1           = 1.6
BEST_B            = 0.75
BM25_WEIGHT       = 0.8

print('Libraries loaded successfully.')

## Step 1 — Load Queries and Relevance Judgements (qrels)

In [ ]:
print(f'Loading dataset: {DATASET_NAME} ...')
dataset = ir_datasets.load(DATASET_NAME)

queries = {q.query_id: q.text for q in dataset.queries_iter()}
print(f'  Queries loaded: {len(queries):,}')

qrels_binary = {}   # for MAP / Recall / P@10  (relevant if relevance > 0)
qrels_graded = {}   # for nDCG (keep original relevance score)

for qrel in dataset.qrels_iter():
    qid, did, rel = qrel.query_id, qrel.doc_id, qrel.relevance
    if rel > 0:
        qrels_binary.setdefault(qid, set()).add(did)
    qrels_graded.setdefault(qid, {})[did] = rel

print(f'  Queries with qrels: {len(qrels_binary):,}')

valid_qids = [qid for qid in queries if qid in qrels_binary]
np.random.seed(42)
if QUERY_SAMPLE_SIZE and len(valid_qids) > QUERY_SAMPLE_SIZE:
    sample_qids = list(np.random.choice(valid_qids, QUERY_SAMPLE_SIZE, replace=False))
    print(f'  Using random sample of {QUERY_SAMPLE_SIZE} queries (seed=42).')
else:
    sample_qids = valid_qids
    print(f'  Using all {len(sample_qids):,} queries.')

## Step 2 — Metric Functions

In [ ]:
def precision_at_k(retrieved, relevant, k=10):
    if not retrieved or not relevant:
        return 0.0
    return sum(1 for doc in retrieved[:k] if doc in relevant) / k


def recall_at_k(retrieved, relevant, k=10):
    if not retrieved or not relevant:
        return 0.0
    return sum(1 for doc in retrieved[:k] if doc in relevant) / len(relevant)


def average_precision(retrieved, relevant):
    if not retrieved or not relevant:
        return 0.0
    hits, sum_p = 0, 0.0
    for i, doc in enumerate(retrieved):
        if doc in relevant:
            hits += 1
            sum_p += hits / (i + 1)
    return sum_p / len(relevant)


def ndcg_at_k(retrieved, graded_rels, k=10):
    """
    nDCG@k — Normalized Discounted Cumulative Gain.
    graded_rels: dict {doc_id: relevance_score}
    """
    if not retrieved or not graded_rels:
        return 0.0
    dcg  = sum(graded_rels.get(doc, 0) / np.log2(i + 2) for i, doc in enumerate(retrieved[:k]))
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(sorted(graded_rels.values(), reverse=True)[:k]))
    return dcg / idcg if idcg > 0 else 0.0


def compute_all_metrics(retrieved, relevant_binary, graded_rels, k=10):
    return {
        'AP':      average_precision(retrieved, relevant_binary),
        'P@10':    precision_at_k(retrieved, relevant_binary, k),
        'Recall':  recall_at_k(retrieved, relevant_binary, k),
        'nDCG@10': ndcg_at_k(retrieved, graded_rels, k),
    }

print('Metric functions defined.')

## Step 3 — Evaluation Function

In [ ]:
def evaluate_model(model_type, query_ids, enable_ner=False, enable_cluster=False,
                   hybrid_type='parallel', k=TOP_K):
    all_metrics = []
    label = model_type if model_type != 'hybrid' else f'hybrid/{hybrid_type}'

    for qid in tqdm(query_ids, desc=f'  {label}', leave=False):
        q_text   = queries.get(qid)
        relevant = qrels_binary.get(qid, set())
        graded   = qrels_graded.get(qid, {})
        if not q_text or not relevant:
            continue

        payload = {
            'dataset_name':               DATASET_NAME,
            'query':                      q_text,
            'model_type':                 model_type,
            'top_k':                      k,
            'k1':                         BEST_K1,
            'b':                          BEST_B,
            'hybrid_bm25_weight':         BM25_WEIGHT,
            'hybrid_type':                hybrid_type,
            'enable_ner_reranking':       enable_ner,
            'enable_cluster_reranking':   enable_cluster,
        }

        try:
            resp = requests.post(SEARCH_API_URL, json=payload, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            retrieved = [r['doc_id'] for r in (data if isinstance(data, list) else data.get('results', []))]
        except Exception:
            retrieved = []

        all_metrics.append(compute_all_metrics(retrieved, relevant, graded, k))

    if not all_metrics:
        return {'MAP': 0.0, 'P@10': 0.0, 'Recall@10': 0.0, 'nDCG@10': 0.0}

    df = pd.DataFrame(all_metrics)
    return {
        'MAP':       round(df['AP'].mean(), 4),
        'P@10':      round(df['P@10'].mean(), 4),
        'Recall@10': round(df['Recall'].mean(), 4),
        'nDCG@10':   round(df['nDCG@10'].mean(), 4),
    }

print('evaluate_model() ready.')

## Step 4 — Phase 1: Evaluation WITHOUT Additional Features

In [ ]:
print('=' * 60)
print('PHASE 1: Evaluation WITHOUT additional features')
print('  NER reranking = OFF | Cluster reranking = OFF')
print('=' * 60)

models_config = [
    ('tfidf',  'parallel'),
    ('bm25',   'parallel'),
    ('bert',   'parallel'),
    ('hybrid', 'parallel'),
    ('hybrid', 'serial'),
]

results_before = []
for model_type, h_type in models_config:
    label = model_type if model_type != 'hybrid' else f'hybrid/{h_type}'
    print(f'\nEvaluating: {label} ...')
    m = evaluate_model(model_type=model_type, query_ids=sample_qids,
                       enable_ner=False, enable_cluster=False, hybrid_type=h_type)
    m['Model'] = label
    results_before.append(m)
    print(f'  -> MAP={m["MAP"]:.4f}  P@10={m["P@10"]:.4f}  Recall@10={m["Recall@10"]:.4f}  nDCG@10={m["nDCG@10"]:.4f}')

df_before = pd.DataFrame(results_before).set_index('Model')
print('\nPhase 1 complete.')

In [ ]:
print('\n--- Results BEFORE additional features ---')
display(df_before.style
        .format('{:.4f}')
        .background_gradient(cmap='YlGn', axis=0)
        .set_caption('Evaluation before additional features — beir/quora/test'))

## Step 5 — Phase 2: Evaluation WITH Additional Features

In [ ]:
print('=' * 60)
print('PHASE 2: Evaluation WITH additional features')
print('  NER reranking = ON | Cluster reranking = ON')
print('=' * 60)

results_after = []
for model_type, h_type in models_config:
    label = model_type if model_type != 'hybrid' else f'hybrid/{h_type}'
    print(f'\nEvaluating: {label} ...')
    m = evaluate_model(model_type=model_type, query_ids=sample_qids,
                       enable_ner=True, enable_cluster=True, hybrid_type=h_type)
    m['Model'] = label
    results_after.append(m)
    print(f'  -> MAP={m["MAP"]:.4f}  P@10={m["P@10"]:.4f}  Recall@10={m["Recall@10"]:.4f}  nDCG@10={m["nDCG@10"]:.4f}')

df_after = pd.DataFrame(results_after).set_index('Model')
print('\nPhase 2 complete.')

In [ ]:
print('\n--- Results AFTER additional features ---')
display(df_after.style
        .format('{:.4f}')
        .background_gradient(cmap='YlGn', axis=0)
        .set_caption('Evaluation after additional features — beir/quora/test'))

## Step 6 — Full Comparison Table (Before vs After)

In [ ]:
cols_before = {c: c + ' (before)' for c in df_before.columns}
cols_after  = {c: c + ' (after)'  for c in df_after.columns}
df_compare  = df_before.rename(columns=cols_before).join(df_after.rename(columns=cols_after))

ordered = []
for metric in ['MAP', 'P@10', 'Recall@10', 'nDCG@10']:
    ordered += [metric + ' (before)', metric + ' (after)']
df_compare = df_compare[ordered]

print('--- Full Comparison: Before vs After Additional Features ---')
display(df_compare.style
        .format('{:.4f}')
        .background_gradient(cmap='coolwarm', axis=None)
        .set_caption('Model comparison before and after additional features — beir/quora/test'))

## Step 7 — Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison — beir/quora/test', fontsize=14, fontweight='bold')

metrics       = ['MAP', 'P@10', 'Recall@10', 'nDCG@10']
model_labels  = df_before.index.tolist()
x = np.arange(len(model_labels))
w = 0.35

for ax, metric in zip(axes.flatten(), metrics):
    vals_b = df_before[metric].values
    vals_a = df_after[metric].values
    bars_b = ax.bar(x - w/2, vals_b, w, label='Before features', color='#4C8BE0', alpha=0.85)
    bars_a = ax.bar(x + w/2, vals_a, w, label='After features',  color='#E07B4C', alpha=0.85)
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0, max(max(vals_b), max(vals_a)) * 1.25 + 0.01)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    for bar in list(bars_b) + list(bars_a):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../evaluation_results_quora.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to evaluation_results_quora.png')

## Step 8 — Improvement Delta (%)

In [ ]:
df_delta = pd.DataFrame(index=df_before.index)
for metric in ['MAP', 'P@10', 'Recall@10', 'nDCG@10']:
    b = df_before[metric]
    a = df_after[metric]
    df_delta[metric + ' Δ%'] = ((a - b) / b.replace(0, np.nan) * 100).round(2)

print('--- Improvement after additional features (%) ---')
display(df_delta.style
        .format('{:+.2f}%')
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption('Δ% = (after − before) / before × 100  |  Green = improved, Red = declined'))

## Analysis Summary

### Model observations:
1. **TF-IDF** — Classic vector space model. Fast retrieval, reasonable precision for keyword-heavy queries.
2. **BM25** — Outperforms TF-IDF by accounting for document length and term saturation.
3. **BERT** — Best at understanding semantic meaning. Slower but captures context TF-IDF/BM25 miss.
4. **Hybrid Parallel** — Combines BM25 + BERT via weighted score fusion. Balances keyword and semantic matching.
5. **Hybrid Serial** — BM25 retrieves top-100 candidates, BERT re-ranks them. Higher precision with lower compute cost than full hybrid.

### Effect of additional features:
- **NER Reranking** — Boosts documents matching named entities (people, places) in the query.
- **Cluster Reranking** — Prioritizes documents from the topically closest cluster to the query.